# Notebook 3. MongoDB Development and Query Optimisation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/D1lxrry/Northstar-Task/blob/main/colab_notebooks/03_mongodb_and_query_optimisation.ipynb)

The document-paradigm side of the NorthStar Urban Mobility & Logistics coursework. Builds the embedded 
`orders_aggregate` collection, runs the 12 analytical aggregations the 
MongoDB band rewards, then profiles 4 query archetypes before and after 
indexing.

The notebook tries my live free-tier Atlas cluster first. If Atlas is 
paused or unreachable, it falls back to `mongomock` (an in-memory MongoDB 
implementation) so every cell still runs to completion. The marker can 
press *Runtime > Run all* and see results regardless of cluster state.

**Sections**
1. Setup (Atlas-with-mongomock-fallback)
2. Bulk load the 9 raw CSVs into 9 collections
3. Starter queries Q1 to Q6 (incl. zone-quality discovery)
4. Zone canonicalisation cleanup
5. Build the embedded `orders_aggregate` collection
6. Analytical queries v1 to v6 (incl. $lookup)
7. Query optimisation: 4 archetypes, before vs after
8. Recommended production index set
9. Cross-paradigm validation
10. Rubric coverage table

## 1. Setup (Atlas with mongomock fallback)

In [ ]:
# Install the 2 Mongo libraries. mongomock gives us an in-memory MongoDB
# so the notebook runs even when Atlas is paused or behind a firewall.
%pip install --quiet 'pymongo[srv]==4.6.1' mongomock==4.1.2

In [ ]:
import io, csv, sys, time, traceback
from datetime import datetime
import pandas as pd
import urllib.request

RAW = 'https://raw.githubusercontent.com/D1lxrry/Northstar-Task/main'

# Try Atlas first; if it fails for any reason, swap to mongomock.
ATLAS_URI = (
    'mongodb+srv://larryco211_db_user:ud9FGTzMgygYgBVb'
    '@northstar.qltsnwd.mongodb.net/?appName=NorthStar'
)

def get_client():
    # Atlas creds in this notebook are read-only by design, so any write
    # call (drop, insert, build) would fail. We ping first, then probe a
    # tiny write; if either step fails we fall back to mongomock so the
    # rest of the notebook (drops, bulk loads, $out) runs end to end.
    try:
        from pymongo import MongoClient
        c = MongoClient(ATLAS_URI, serverSelectionTimeoutMS=5000)
        c.admin.command('ping')
        probe = c['__write_probe__']['probe']
        probe.insert_one({'_t': 'probe'})
        probe.drop()
        print('connected to live Atlas cluster (read-write)')
        return c, 'atlas'
    except Exception as exc:
        print(f'Atlas not usable for writes ({type(exc).__name__}); falling back to mongomock')
        import mongomock
        return mongomock.MongoClient(), 'mongomock'

client, backend = get_client()
db = client['northstar']
# In mongomock mode the database starts empty. In Atlas mode the previous
# work is still there; drop everything for a clean rerun either way.
for c in db.list_collection_names():
    db[c].drop()
print(f'backend = {backend}')

## 2. Bulk load the 9 raw CSVs into 9 collections

Each CSV becomes one collection. A shared `coerce` helper turns string 
values into native ints, floats, datetimes or bools so the aggregation 
operators downstream behave correctly.

In [ ]:
DATE_FORMATS = [
    '%Y-%m-%dT%H:%M:%S.%f', '%Y-%m-%dT%H:%M:%S',
    '%Y-%m-%d %H:%M:%S.%f', '%Y-%m-%d %H:%M:%S', '%Y-%m-%d',
]

def coerce(value):
    if value is None: return None
    s = str(value).strip()
    if s == '': return None
    if s.lstrip('-').isdigit():
        try: return int(s)
        except ValueError: pass
    try: return float(s)
    except ValueError: pass
    for fmt in DATE_FORMATS:
        try: return datetime.strptime(s, fmt)
        except ValueError: continue
    if s.lower() in ('true','false'):
        return s.lower() == 'true'
    return s

def load_csv_to_collection(name):
    url = f'{RAW}/northstar_dataset/{name}.csv'
    text = urllib.request.urlopen(url).read().decode('utf-8')
    reader = csv.DictReader(io.StringIO(text))
    docs = [{k: coerce(v) for k, v in row.items()} for row in reader]
    if docs: db[name].insert_many(docs)
    return len(docs)

for n in ['customers','orders','deliveries','drivers','vehicles',
          'hubs','incidents','complaints','app_events']:
    print(f'{n:<12} {load_csv_to_collection(n):>5} docs')

## 3. Starter queries Q1 to Q6

These exercise the basic aggregation idioms the rubric calls out: 
`$group`, `$sort`, `$lookup`, `distinct`. Q4 is where the zone-quality 
issue surfaced (Central, central, CENTRAL, Ctr all showing as separate 
groups); the fix lives in section 4.

In [ ]:
def show(title, rows, limit=8):
    print(f'\n{title}'); print('-' * len(title))
    for i, r in enumerate(rows):
        if i >= limit: print(f'  ... and more'); break
        print(' ', r)

# Q1. orders by priority_level
show('Q1. orders by priority_level',
     db.orders.aggregate([
         {'$group': {'_id': '$priority_level', 'n': {'$sum': 1}}},
         {'$sort': {'n': -1}}]))

# Q2. complaints by complaint_type
show('Q2. complaints by complaint_type',
     db.complaints.aggregate([
         {'$group': {'_id': '$complaint_type', 'n': {'$sum': 1}}},
         {'$sort': {'n': -1}}]))

# Q3. revenue per service_type
show('Q3. revenue per service_type',
     db.orders.aggregate([
         {'$group': {'_id': '$service_type',
                     'orders': {'$sum': 1},
                     'total_revenue': {'$sum': '$order_value'},
                     'avg_value':     {'$avg': '$order_value'}}},
         {'$sort': {'total_revenue': -1}}]))

In [ ]:
# Q4. distinct pickup_zone values (the data quality discovery)
zones = sorted({z for z in db.orders.distinct('pickup_zone') if z})
print('\nQ4. distinct pickup_zone values')
print(f'  {len(zones)} distinct values: {zones}')

# Q5. $lookup orders to deliveries, then filter to failed ones
show('Q5. orders joined to deliveries (failed only, $lookup)',
     db.orders.aggregate([
         {'$lookup': {'from': 'deliveries',
                      'localField': 'order_id',
                      'foreignField': 'order_id', 'as': 'delivery'}},
         {'$unwind': '$delivery'},
         {'$match': {'delivery.delivery_status': {'$regex': '^failed$', '$options': 'i'}}},
         {'$project': {'_id': 0, 'order_id': 1, 'service_type': 1,
                       'pickup_zone': 1,
                       'status': '$delivery.delivery_status'}},
         {'$limit': 5}]),
     limit=5)

# Q6. distinct delivery_status (the case-spelling diagnostic)
show('Q6. distinct delivery_status (case-spelling diagnostic)',
     db.deliveries.aggregate([
         {'$group': {'_id': '$delivery_status', 'n': {'$sum': 1}}},
         {'$sort': {'n': -1}}]))

## 4. Zone canonicalisation cleanup

Q4 surfaces 16 raw zone spellings that should be 7 canonical zones. The 
fix is a shared `ZONE_MAP` applied with `updateMany` to every 
zone-bearing field across every collection. The R and Python pipelines 
use the same map so the 3 paradigms agree downstream.

In [ ]:
ZONE_MAP = {
    'AIRPORT': 'Airport',     'Airport': 'Airport',
    'CENTRAL': 'Central',     'Central': 'Central',     'Ctr': 'Central',
    'EAST': 'East',           'East': 'East',
    'NORTH': 'North',         'North': 'North',         'north': 'North',
    'RiverSide': 'Riverside', 'Riverside': 'Riverside',
    'SOUTH': 'South',         'South': 'South',
    'WEST': 'West',           'West': 'West',
}
ZONE_FIELDS = [
    ('orders', ['pickup_zone', 'dropoff_zone']),
    ('customers', ['home_zone']),
    ('drivers', ['base_zone']),
    ('vehicles', ['assigned_zone']),
    ('hubs', ['zone']),
    ('app_events', ['zone_context']),
]
for collection_name, fields in ZONE_FIELDS:
    for raw, canon in ZONE_MAP.items():
        if raw == canon: continue
        for field in fields:
            db[collection_name].update_many({field: raw}, {'$set': {field: canon}})

zones_now = sorted({z for z in db.orders.distinct('pickup_zone') if z})
print(f'after cleanup, {len(zones_now)} canonical zones: {zones_now}')

## 5. Build the embedded `orders_aggregate` collection

Order-centric document: customer snapshot embedded, delivery embedded 
(with incidents nested inside), complaints and app_events embedded as 
arrays. Drivers, vehicles and hubs stay as referenced collections.

In [ ]:
db['orders_aggregate'].drop()

# Cache child docs grouped by parent key so we do not re-query per order.
customers_by_id = {c['customer_id']: c for c in db.customers.find()}
deliveries_by_order = {d['order_id']: d for d in db.deliveries.find()}
incidents_by_delivery = {}
for i in db.incidents.find():
    incidents_by_delivery.setdefault(i['delivery_id'], []).append(i)
complaints_by_order = {}
for c in db.complaints.find():
    complaints_by_order.setdefault(c['order_id'], []).append(c)
events_by_order = {}
for e in db.app_events.find():
    events_by_order.setdefault(e.get('order_id'), []).append(e)

def strip_id(d):
    out = dict(d); out.pop('_id', None); return out

agg_docs = []
for o in db.orders.find():
    o = strip_id(o)
    cust = customers_by_id.get(o['customer_id'])
    deliv = deliveries_by_order.get(o['order_id'])
    embedded = {
        **o,
        'customer': strip_id(cust) if cust else None,
        'delivery': None,
        'complaints': [strip_id(c) for c in complaints_by_order.get(o['order_id'], [])],
        'app_events': [strip_id(e) for e in events_by_order.get(o['order_id'], [])],
    }
    if deliv:
        deliv = strip_id(deliv)
        deliv['incidents'] = [strip_id(i) for i in
                              incidents_by_delivery.get(deliv['delivery_id'], [])]
        embedded['delivery'] = deliv
    agg_docs.append(embedded)

if agg_docs: db['orders_aggregate'].insert_many(agg_docs)
print(f'orders_aggregate: {db.orders_aggregate.count_documents({})} documents')

## 6. Analytical queries v1 to v6 (incl. $lookup)

These run against the embedded `orders_aggregate` collection. Most 
are 1-document reads because the schema embeds delivery, customer 
and complaints. The driver-performance query (v5) is the only one 
that needs `$lookup` because drivers are intentionally not embedded.

In [ ]:
oa = db['orders_aggregate']

show('v1. failure rate by service_type',
     oa.aggregate([
         {'$match': {'delivery.delivery_status': {'$ne': None}}},
         {'$group': {'_id': '$service_type',
                     'orders': {'$sum': 1},
                     'failed': {'$sum': {'$cond': [
                         {'$eq': ['$delivery.delivery_status', 'Failed']}, 1, 0]}}}},
         {'$project': {'orders': 1, 'failed': 1,
                       'failure_rate': {'$divide': ['$failed', '$orders']}}},
         {'$sort': {'failure_rate': -1}}]))

In [ ]:
show('v2. avg rating by pickup_zone',
     oa.aggregate([
         {'$match': {'delivery.customer_rating_post_delivery': {'$ne': None}}},
         {'$group': {'_id': '$pickup_zone',
                     'avg_rating': {'$avg': '$delivery.customer_rating_post_delivery'},
                     'n': {'$sum': 1}}},
         {'$sort': {'avg_rating': -1}}]))

In [ ]:
show('v3. top compound-risk orders (complaints + failed + low rating)',
     oa.aggregate([
         {'$match': {'delivery.delivery_status': 'Failed',
                     'delivery.customer_rating_post_delivery': {'$lt': 3.0},
                     'complaints.0': {'$exists': True}}},
         {'$project': {'_id': 0, 'order_id': 1, 'service_type': 1,
                       'pickup_zone': 1,
                       'rating': '$delivery.customer_rating_post_delivery',
                       'complaint_types': '$complaints.complaint_type'}},
         {'$sort': {'rating': 1}},
         {'$limit': 5}]))

In [ ]:
show('v4. zone incident density',
     oa.aggregate([
         {'$match': {'delivery': {'$ne': None}}},
         {'$group': {'_id': '$pickup_zone',
                     'deliveries': {'$sum': 1},
                     'incidents': {'$sum': {'$size': {'$ifNull': ['$delivery.incidents', []]}}}}},
         {'$project': {'deliveries': 1, 'incidents': 1,
                       'incidents_per_delivery':
                           {'$divide': ['$incidents', '$deliveries']}}},
         {'$sort': {'incidents_per_delivery': -1}}]))

In [ ]:
show('v5. top 10 drivers by completed deliveries ($lookup join)',
     oa.aggregate([
         {'$match': {'delivery.delivery_status': 'Success',
                     'delivery.driver_id': {'$ne': None}}},
         {'$group': {'_id': '$delivery.driver_id',
                     'completed': {'$sum': 1},
                     'avg_rating': {'$avg': '$delivery.customer_rating_post_delivery'}}},
         {'$sort': {'completed': -1}},
         {'$limit': 10},
         {'$lookup': {'from': 'drivers',
                      'localField': '_id',
                      'foreignField': 'driver_id', 'as': 'drv'}},
         {'$unwind': '$drv'},
         {'$project': {'_id': 0, 'driver_id': '$_id',
                       'base_zone': '$drv.base_zone',
                       'completed': 1,
                       'avg_rating': {'$round': ['$avg_rating', 2]}}}]))

In [ ]:
show('v6. app-engagement bucket vs failure rate',
     oa.aggregate([
         {'$match': {'delivery': {'$ne': None},
                     'customer.app_engagement_score': {'$ne': None}}},
         {'$bucket': {'groupBy': '$customer.app_engagement_score',
                      'boundaries': [0, 30, 60, 100],
                      'default': 'unknown',
                      'output': {'orders': {'$sum': 1},
                                 'failed': {'$sum': {'$cond': [
                                     {'$eq': ['$delivery.delivery_status', 'Failed']},
                                     1, 0]}}}}}]))

## 7. Query optimisation: 4 archetypes, before vs after

For each archetype: drop secondary indexes (baseline COLLSCAN), run 
`explain('executionStats')`, build the proposed index, rerun, compute 
the docsExamined reduction factor. On mongomock the explain output is 
simulated; if you are on the live Atlas backend you will see real 
executionStats numbers and stage chains.

In [ ]:
ARCHETYPES = [
    {'id': 'Q_A', 'name': 'Single-field equality on service_type',
     'filter': {'service_type': 'Business'},
     'index':  [('service_type', 1)]},
    {'id': 'Q_B', 'name': 'Compound predicate (service_type + delivery.delivery_status)',
     'filter': {'service_type': 'Business', 'delivery.delivery_status': 'Failed'},
     'index':  [('service_type', 1), ('delivery.delivery_status', 1)]},
    {'id': 'Q_C', 'name': 'High-cardinality point lookup on order_id',
     'filter': {'order_id': 'O00023'},
     'index':  [('order_id', 1)]},
    {'id': 'Q_D', 'name': 'Mid-cardinality equality on pickup_zone',
     'filter': {'pickup_zone': 'Central'},
     'index':  [('pickup_zone', 1)]},
]

def drop_secondary_indexes(coll):
    for ix in coll.list_indexes():
        if ix['name'] != '_id_':
            try: coll.drop_index(ix['name'])
            except Exception: pass

def stats_for(coll, flt):
    # Atlas branch: real explain via db.command. PyMongo's
    # Cursor.explain() takes no arguments, so we use the explicit
    # command form to request the executionStats verbosity.
    if backend == 'atlas':
        try:
            plan = db.command({
                'explain': {'find': coll.name, 'filter': flt},
                'verbosity': 'executionStats',
            })
            es = plan.get('executionStats', {})
            win = plan.get('queryPlanner', {}).get('winningPlan', {})
            return {
                'stage': win.get('stage'),
                'nReturned': es.get('nReturned', 0),
                'docsExamined': es.get('totalDocsExamined', 0),
                'keysExamined': es.get('totalKeysExamined', 0),
                'timeMs': es.get('executionTimeMillis', 0),
            }
        except Exception as exc:
            print(f'  atlas explain failed ({type(exc).__name__}: {exc}); using simulated stats')
    # Mongomock fallback (or atlas explain failure): simulate stats
    # by counting matches vs total and checking which indexes exist.
    n = coll.count_documents(flt)
    total = coll.count_documents({})
    has_idx = any(
        sorted(ix.get('key', {}).items()) == sorted(flt.keys())
        for ix in coll.list_indexes()
    ) or any(all(k in ix.get('key', {}) for k in flt.keys())
             for ix in coll.list_indexes() if ix['name'] != '_id_')
    return {
        'stage': 'IXSCAN (simulated)' if has_idx else 'COLLSCAN (simulated)',
        'nReturned': n,
        'docsExamined': n if has_idx else total,
        'keysExamined': n if has_idx else 0,
        'timeMs': 0,
    }

results = []
for a in ARCHETYPES:
    drop_secondary_indexes(oa)
    before = stats_for(oa, a['filter'])
    oa.create_index(a['index'], name=f"idx_{a['id']}")
    after = stats_for(oa, a['filter'])
    ratio = before['docsExamined'] / max(after['docsExamined'], 1)
    results.append({
        'id': a['id'],
        'name': a['name'],
        'stage_before': before['stage'],
        'stage_after': after['stage'],
        'docs_before': before['docsExamined'],
        'docs_after': after['docsExamined'],
        'reduction': round(ratio, 1),
    })

qopt = pd.DataFrame(results)
print(qopt.to_string(index=False))

## 8. Recommended production index set

Drop the experimental indexes from section 7 and install the 5 indexes 
the Query Optimisation Report recommends for production deployment.

In [ ]:
drop_secondary_indexes(oa)
PROD_INDEXES = [
    ([('order_id', 1)],                                {'unique': True, 'name': 'order_id_unique'}),
    ([('service_type', 1), ('delivery.delivery_status', 1)], {'name': 'service_type_delivery_status'}),
    ([('pickup_zone', 1)],                            {'name': 'pickup_zone_1'}),
    ([('delivery.driver_id', 1)],                     {'name': 'delivery_driver_id_1'}),
    ([('customer.customer_id', 1)],                   {'name': 'customer_customer_id_1'}),
]
for keys, opts in PROD_INDEXES:
    try:
        nm = oa.create_index(keys, **opts)
        print(f'  + {nm}: {dict(keys)}')
    except Exception as exc:
        print(f'  ! {opts.get("name")}: {exc}')

print('\nfinal index set on orders_aggregate:')
for ix in oa.list_indexes():
    print(f'  - {ix["name"]:<35} keys={dict(ix["key"])}')

## 9. Cross-paradigm validation

The aggregations in section 6 return the same numbers as the SQL 
queries in Notebook 2 (S1 to S8). The chi square on pickup_zone vs 
delivery_status is computable directly here (the contingency table 
comes from a single `$group`) and matches Notebook 1's scipy result 
to 4 decimal places.

In [ ]:
import collections
tab = collections.Counter()
for d in oa.find({}, {'pickup_zone': 1, 'delivery.delivery_status': 1}):
    s = (d.get('delivery') or {}).get('delivery_status')
    z = d.get('pickup_zone')
    if s and z: tab[(z, s)] += 1

ct = pd.Series(tab).unstack(fill_value=0).sort_index()
print('Pickup zone x delivery status contingency table:')
print(ct)

try:
    from scipy import stats
    chi2, p, dof, _ = stats.chi2_contingency(ct.values)
    print(f'\nchi2 = {chi2:.2f}, dof = {dof}, p = {p:.4f}')
except Exception as exc:
    print(f'(scipy not available: {exc})')

## 10. Rubric coverage

In [ ]:
rubric = pd.DataFrame([
    ['2. Bulk load',          'PyMongo coerce + insert_many for 9 CSVs',  'MongoDB development (20)'],
    ['3. Starter queries',    '$group, $sort, $lookup, distinct',          'MongoDB development (20)'],
    ['4. Zone canonicalisation', 'updateMany across every zone field',     'MongoDB development (20)'],
    ['5. orders_aggregate',   'Embedded order document, 1 doc per order',  'MongoDB development (20)'],
    ['6. Analytical queries', '12 aggregations covering all required idioms', 'MongoDB development (20)'],
    ['7. 4-archetype experiment','docsExamined before vs after indexing',   'Query optimisation (10)'],
    ['8. Production index set', '5 indexes recommended in Section 7 report','Query optimisation (10)'],
    ['9. Cross-paradigm validation', 'Chi square matches Python and R',    'MongoDB development (20)'],
], columns=['Section', 'What it shows', 'Rubric line'])
print(rubric.to_string(index=False))